# Mathematics Applications

Use this notebook when the **graph family itself** is the object you want to study.

Unlike the physics notebook, you do not need to start from a Hamiltonian or a sampled
surface. You can construct a graph directly, compute its Yamada polynomial, compare
families, and generate exact datasets.

A new user can follow this path:

1. construct one graph and inspect it;
2. compute its Yamada polynomial;
3. compare two evaluation routes;
4. scan one graph family over a parameter;
5. inspect a structured catalog case;
6. use PD codes as an interface for custom knot/link invariants; and
7. generate a larger dataset only after the single-graph workflow is clear.

The key distinction is:

- **graph construction** specifies the mathematical object;
- **embedded/projection workflows** include geometric crossing information;
- **abstract graph-family calculations** study the combinatorial family directly.

PD-code workflows can also act as an interoperability layer: once a spatial embedding has been projected, the resulting diagram data can be passed to user-defined knot/link invariants without modifying the core library.

## 1. Set up the mathematical examples

Run these cells once to import SymPy, NetworkX, the Yamada routines, and the structured
graph builders used below.

In [ ]:
from pathlib import Path
import sys
import importlib.util
import os
import tempfile

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "Could not locate the KnottedGraph repository root. "
        "Run this notebook from inside the repository checkout."
    )

DOC_ROOT = PROJECT_ROOT / "doc"

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "knottedgraph-mpl"))

for package in ["numpy", "networkx", "sympy", "plotly", "matplotlib", "pyvista"]:
    print(f"{package:10s} = {importlib.util.find_spec(package) is not None}")


In [ ]:
import math
import time

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import sympy as sp
from IPython.display import Math, display

import knotted_graph
from knotted_graph.invariants.yamada.native import (
    native_available,
    native_import_error,
)

print("Python executable:", sys.executable)
print("KnottedGraph:", Path(knotted_graph.__file__).resolve())
print("Native Yamada backend:", native_available())
print("Native import error:", native_import_error())

from knotted_graph.projection import (
    PDCode,
    compute_yamada_polynomial,
    sample_projections,
    select_projection,
)
from knotted_graph.visualization import plot_3D_graph_plotly

BLUE = "#1f77b4"
RED = "#d62728"
CAMERA = dict(eye=dict(x=4.0, y=4.0, z=3.0))
pio.renderers.default = "notebook_connected"
Y = sp.Symbol("Y")
kx, ky, kz = sp.symbols("k_x k_y k_z", real=True)


def axis_style():
    return dict(
        visible=True,
        title="",
        showticklabels=False,
        showbackground=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="black",
        linewidth=2,
    )


def apply_kg_layout(fig, *, width=760, height=620):
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
            xaxis=axis_style(),
            yaxis=axis_style(),
            zaxis=axis_style(),
            aspectmode="data",
            camera=CAMERA,
        ),
    )
    return fig


def plot_surface_polydata(surface, *, opacity=0.58):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = mesh.points
    fig = go.Figure(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=BLUE,
            opacity=opacity,
        )
    )
    return apply_kg_layout(fig)


def plot_points_3d(points, *, size=3):
    points = np.asarray(points)
    fig = go.Figure(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=dict(size=size, color=BLUE),
        )
    )
    return apply_kg_layout(fig)


def plot_graph_kg(graph):
    return apply_kg_layout(plot_3D_graph_plotly(graph))


def print_upsilon(label, expr):
    print(f"Upsilon({label}; Y) = {sp.expand(expr)}")


def display_bloch_vector(label, components):
    display(Math(label + r"=" + sp.latex(sp.Matrix(components))))


print("shared plotting and notation helpers ready")

from plotly.subplots import make_subplots


def add_surface_trace(
    fig,
    surface,
    *,
    row=1,
    col=1,
    opacity=0.58,
    color=BLUE,
):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = np.asarray(mesh.points)

    fig.add_trace(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=color,
            opacity=opacity,
            showscale=False,
        ),
        row=row,
        col=col,
    )


def add_graph_traces(
    fig,
    graph,
    *,
    row=1,
    col=1,
):
    graph_fig = (
        plot_3D_graph_plotly(
            graph
        )
    )

    for trace in graph_fig.data:
        fig.add_trace(
            trace,
            row=row,
            col=col,
        )


def style_plotly_scenes(
    fig,
    scene_count,
    *,
    width=980,
    height=660,
):
    for index in range(
        1,
        scene_count + 1,
    ):
        scene_name = (
            "scene"
            if index == 1
            else f"scene{index}"
        )

        fig.update_layout(
            **{
                scene_name: dict(
                    xaxis=axis_style(),
                    yaxis=axis_style(),
                    zaxis=axis_style(),
                    aspectmode="data",
                    camera=CAMERA,
                )
            }
        )

    fig.update_layout(
        width=width,
        height=height,
        margin=dict(
            l=0,
            r=0,
            t=10,
            b=0,
        ),
        showlegend=False,
    )

    return fig


def surface_component_summary(
    surface,
):
    try:
        bodies = (
            surface.split_bodies()
        )

        components = [
            body
            for body in bodies
            if (
                body is not None
                and getattr(
                    body,
                    "n_cells",
                    0,
                ) > 0
            )
        ]

        sizes = sorted(
            [
                (
                    component.n_points,
                    component.n_cells,
                )
                for component
                in components
            ],
            key=lambda item: item[1],
            reverse=True,
        )

        return (
            len(components),
            sizes,
        )

    except Exception:
        return None, []


def print_surface_summary(
    label,
    surface,
):
    n_components, sizes = (
        surface_component_summary(
            surface
        )
    )

    print(
        f"{label}: "
        f"{surface.n_points} points, "
        f"{surface.n_cells} cells"
    )

    if n_components is None:
        print(
            "  connected components = unavailable"
        )

    else:
        print(
            "  connected components =",
            n_components,
        )

        if n_components > 1:
            print(
                "  component sizes "
                "(points, cells) =",
                sizes,
            )


## 2. Compare two Yamada evaluation routes

When the library provides more than one exact evaluation route, comparing them on the
same graph is a useful consistency check.

The two expressions should agree after simplification. If they do not, inspect the graph
construction and the method assumptions before scanning a larger family.

In [ ]:
def mathematical_k4_spine(samples=90, amplitude=0.75):
    vertices = {
        "a": np.array([-1.15, -0.78, -0.38]),
        "b": np.array([1.18, -0.64, 0.30]),
        "c": np.array([0.86, 0.95, -0.26]),
        "d": np.array([-0.88, 0.84, 0.52]),
    }
    edge_specs = [
        ("a", "b", "ab", np.array([0.00, 0.90, 0.70]), 0.0),
        ("a", "c", "ac", np.array([0.35, -0.15, 1.00]), 1.1),
        ("a", "d", "ad", np.array([0.95, 0.15, -0.25]), 2.2),
        ("b", "c", "bc", np.array([-0.90, 0.25, 0.35]), 0.7),
        ("b", "d", "bd", np.array([-0.20, 1.00, -0.60]), 1.7),
        ("c", "d", "cd", np.array([0.10, -0.90, -0.85]), 2.8),
    ]
    s = np.linspace(0.0, 1.0, samples)
    graph = nx.MultiGraph()
    for vertex_id, pos in vertices.items():
        graph.add_node(vertex_id, pos=pos.copy())

    for u, v, key, bend, phase in edge_specs:
        start = vertices[u]
        end = vertices[v]
        chord = end - start
        bend = bend / np.linalg.norm(bend)
        side = np.cross(chord, bend)
        side = side / np.linalg.norm(side)
        envelope = np.sin(np.pi * s)
        pts = (1 - s)[:, None] * start + s[:, None] * end
        pts += amplitude * envelope[:, None] * (
            np.cos(phase + np.pi * s)[:, None] * bend
            + 0.6 * np.sin(2 * np.pi * s + phase)[:, None] * side
        )
        pts[0] = start
        pts[-1] = end
        graph.add_edge(u, v, key=key, pts=pts)

    graph.graph.update(
        graph_id="mathematical_k4",
        input_kind="internal_mathematical_geometry",
        is_closed=True,
    )
    return graph


math_graph = mathematical_k4_spine()
math_projection = select_projection(math_graph, num_rotation_samples=16)

negami = compute_yamada_polynomial(
    math_graph,
    Y,
    rotation_angles=math_projection.rotation_angles,
    method="negami",
    n_jobs=1,
)
recursive = compute_yamada_polynomial(
    math_graph,
    Y,
    rotation_angles=math_projection.rotation_angles,
    method="recursive",
    n_jobs=1,
)
print_upsilon("G_K4, negami", negami)
print_upsilon("G_K4, recursive", recursive)
print("same_result =", sp.expand(negami - recursive) == 0)


## 3. Scan the theta-graph family

A family scan asks how the invariant changes as a discrete graph parameter changes.

The code below builds several $\Theta_s$ graphs, computes the polynomial for each one,
and keeps the parameter value beside the result. Change the scan range to explore a
larger family.

In [ ]:
from knotted_graph.core import ThetaGraph
from knotted_graph.invariants.yamada import compute_graph_yamada_polynomial

for s in range(2, 8):
    theta = ThetaGraph(s)
    upsilon = compute_graph_yamada_polynomial(theta, Y)
    print_upsilon(f"Theta_{s}", upsilon)


## 4. Explore the structured graph catalog

The catalog provides reusable constructors for several graph families.

Start by listing the available family names. Then build one case, inspect its graph
summary and plot, and compute the invariant. This is the easiest way to learn the
parameter convention for a family before launching a large sweep.

In [ ]:
from knotted_graph.applications.mathematical import (
    GRAPH_FAMILY_CATALOG,
    NOTEBOOK_YAMADA_EXAMPLES,
    build_graph_case,
    graph_summary,
    plot_structured_multigraph,
)
from knotted_graph.invariants.yamada import laurent_y_to_sigma_polynomial

sigma = sp.Symbol("sigma")

for family_name, spec in GRAPH_FAMILY_CATALOG.items():
    print(f"{family_name:24s} sample_args={spec.sample_args}  {spec.note}")


### 4.1 Compute and compare polynomial representations

For each catalog example, the next cells record:

- a compact graph summary;
- the Laurent polynomial in `Y`; and
- the converted polynomial in `sigma`.

The plots keep loops and parallel edges visible so you can compare the graph
structure directly with its exact invariant.

In [ ]:
catalog_results = {}
cols = 3
rows = math.ceil(len(NOTEBOOK_YAMADA_EXAMPLES) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
axes = np.asarray(axes).reshape(-1)

for ax, (family_name, args, label) in zip(axes, NOTEBOOK_YAMADA_EXAMPLES):
    graph, pos = build_graph_case(family_name, *args)
    plot_structured_multigraph(
        graph,
        pos,
        family_name=family_name,
        family_args=args,
        ax=ax,
        show=False,
        node_color=RED,
        node_edge_color=RED,
        edge_color=BLUE,
    )
    yamada_y = sp.expand(compute_graph_yamada_polynomial(graph, Y))
    yamada_sigma = laurent_y_to_sigma_polynomial(yamada_y, Y, sigma).as_expr()
    catalog_results[label] = {
        "summary": graph_summary(graph),
        "Y": yamada_y,
        "sigma": sp.expand(yamada_sigma),
    }

for ax in axes[len(NOTEBOOK_YAMADA_EXAMPLES):]:
    ax.set_axis_off()

plt.tight_layout()
plt.show()

for panel_index, label in enumerate(catalog_results, start=1):
    result = catalog_results[label]
    print(f"panel {panel_index}: {label}  {result['summary']}")
    print_upsilon(label, result["Y"])
    print(f"Upsilon_sigma({label}; sigma) = {result['sigma']}")
    print()


### 4.2 Scan a parameterized family

Call the same graph-family builder repeatedly when you want to study how topology
changes with a discrete parameter.

The cylinder example below varies one parameter and stores the resulting
polynomials. You can use the same pattern for your own family by replacing the
builder and parameter range.

In [ ]:
cylinder_scan = []
for cols in range(3, 7):
    graph, pos = build_graph_case("cylinder", 2, cols)
    yamada_y = sp.expand(compute_graph_yamada_polynomial(graph, Y))
    yamada_sigma = laurent_y_to_sigma_polynomial(yamada_y, Y, sigma).as_expr()
    cylinder_scan.append(
        (cols, graph_summary(graph), yamada_y, sp.expand(yamada_sigma))
    )

graph, pos = build_graph_case("cylinder", 2, 6)
plot_structured_multigraph(
    graph,
    pos,
    family_name="cylinder",
    family_args=(2, 6),
    node_color=RED,
    node_edge_color=RED,
    edge_color=BLUE,
)

for cols, summary, yamada_y, yamada_sigma in cylinder_scan:
    label = f"Cylinder(2,{cols})"
    print(f"{label}: {summary}")
    print_upsilon(label, yamada_y)
    print(f"Upsilon_sigma({label}; sigma) = {yamada_sigma}")
    print()


## 5. Inspect any catalog case interactively

`inspect_catalog_case(...)` is a small convenience wrapper around the public catalog
builder and plotting functions.

Pass the family name followed by that family's parameters. The helper then:

1. builds the graph;
2. plots it;
3. prints basic graph information;
4. computes $\Upsilon(G;Y)$; and
5. prints the corresponding $\sigma$ representation.

Use it to verify a few cases manually before generating a dataset.

In [ ]:
def inspect_catalog_case(family_name, *args):
    graph, pos = build_graph_case(family_name, *args)
    plot_structured_multigraph(
        graph,
        pos,
        family_name=family_name,
        family_args=args,
        node_color=RED,
        node_edge_color=RED,
        edge_color=BLUE,
    )
    yamada_y = sp.expand(compute_graph_yamada_polynomial(graph, Y))
    yamada_sigma = sp.expand(
        laurent_y_to_sigma_polynomial(yamada_y, Y, sigma).as_expr()
    )
    print("summary =", graph_summary(graph))
    print_upsilon(f"{family_name}{args}", yamada_y)
    print(f"Upsilon_sigma({family_name}{args}; sigma) = {yamada_sigma}")
    return graph, yamada_y, yamada_sigma


# Example:
inspect_catalog_case("cylinder", 2, 5)


## 6. Test a Petersen-minor certificate for intrinsic linkedness

Graph-minor questions belong naturally in the mathematical workflow because they concern
the **abstract graph**, rather than the physical field or surface that produced a
particular embedding.

The example below starts from the extracted “Awesome” graph and asks whether it contains
the Petersen graph as a minor.

Interpret the result as follows:

- `True` gives a Petersen-minor certificate. Since the Petersen graph belongs to the
  Petersen family of forbidden minors for linkless embeddings, this is sufficient to
  establish intrinsic linkedness of the abstract graph.
- `False` does **not** establish linkless embeddability. It only says that this particular
  Petersen target was not found; other Petersen-family minors would still need to be
  checked for a complete forbidden-minor test.

The code also returns the branch sets used for the detected minor, when available.

The built-in nodal constructors define a two-band non-Hermitian Hamiltonian from a
complex polynomial $f(z,w)$.  They first return the Bloch vector

$$
\mathbf d_f(\mathbf k)
=
\left(
\operatorname{Re}f,\,
i\gamma,\,
\operatorname{Im}f
\right),
$$

and `NodalSkeleton` constructs

$$
H_f(\mathbf k;\gamma)
=
\mathbf d_f(\mathbf k)\cdot\boldsymbol{\sigma}
=
\begin{pmatrix}
\operatorname{Im}f & \operatorname{Re}f+\gamma\\
\operatorname{Re}f-\gamma & -\operatorname{Im}f
\end{pmatrix}.
$$

Unless a model explicitly overrides them,

$$
z
=
\cos(2k_z)+c
+i\left(
\cos k_x+\cos k_y+\cos k_z-m
\right),
\qquad
w
=
\sin k_x+i\sin k_y.
$$

For the “Awesome” model,

$$
f_{\mathrm{Awesome}}(z,w)
=
z\left(z^2-w^4+w\right).
$$

The extracted graph in this example uses $\gamma=0.2$ and $c=0.5$ before the abstract
minor test is applied.

In [ ]:
from knotted_graph.applications.nodal import NodalSkeleton
from knotted_graph.applications.nodal.models import (
    awesome_bloch_vector,
    threelink_bloch_vector,
)


awesome_minor_ske = NodalSkeleton(
    awesome_bloch_vector(
        0.2,
        k_symbols=(kx, ky, kz),
        c=0.5,
    ),
    k_symbols=(kx, ky, kz),
    dimension=300,
    axis_scale=(1.0, 1.0, 1.5),
)

awesome_minor_graph = (
    awesome_minor_ske.skeleton_graph(
        simplify=True,
        smooth_epsilon=2,
    )
)

petersen_graph = nx.petersen_graph()

minor_embedding = (
    awesome_minor_ske.check_minor(
        petersen_graph,
        awesome_minor_graph,
    )
)

print(
    "awesome graph nodes / edges =",
    (
        awesome_minor_graph.number_of_nodes(),
        awesome_minor_graph.number_of_edges(),
    ),
)
print(
    "degree sequence =",
    sorted(
        dict(
            awesome_minor_graph.degree()
        ).values()
    ),
)
print(
    "Petersen minor found =",
    bool(minor_embedding),
)

if minor_embedding:
    print(
        "branch-set sizes =",
        {
            node: len(branch_set)
            for node, branch_set
            in minor_embedding.items()
        },
    )


# Inspect the extracted embedded graph.
plot_graph_kg(
    awesome_minor_graph
).show()


# Inspect the target Petersen graph separately.
fig, ax = plt.subplots(
    figsize=(5.4, 5.0)
)

petersen_pos = nx.spring_layout(
    petersen_graph,
    seed=4,
)

nx.draw_networkx_edges(
    petersen_graph,
    pos=petersen_pos,
    ax=ax,
    edge_color=BLUE,
    width=2.0,
)

nx.draw_networkx_nodes(
    petersen_graph,
    pos=petersen_pos,
    ax=ax,
    node_color=RED,
    node_size=180,
)

ax.set_title(
    "Target Petersen graph"
)
ax.set_aspect("equal")
ax.axis("off")
plt.show()


## 7. Track planarity and connectivity during a parameter sweep

After the intrinsic-linkedness example, it is natural to inspect additional
**abstract graph diagnostics** across a parameter family.

For each parameter value, this example:

1. keeps the complete extracted surface;
2. extracts the embedded spatial graph;
3. records graph connectivity information; and
4. tests whether the underlying abstract graph is planar.

The distinction is important:

- **graph connectivity** concerns how the abstract graph is connected;
- **abstract planarity** asks whether that graph can be drawn in the plane without
  crossings;
- the topology of the specific embedding in $\mathbb{R}^3$ is a separate question.

Therefore a planar abstract graph is not automatically a topologically trivial
three-dimensional embedding.

The built-in nodal constructors define a two-band non-Hermitian Hamiltonian from a
complex polynomial $f(z,w)$.  They first return the Bloch vector

$$
\mathbf d_f(\mathbf k)
=
\left(
\operatorname{Re}f,\,
i\gamma,\,
\operatorname{Im}f
\right),
$$

and `NodalSkeleton` constructs

$$
H_f(\mathbf k;\gamma)
=
\mathbf d_f(\mathbf k)\cdot\boldsymbol{\sigma}
=
\begin{pmatrix}
\operatorname{Im}f & \operatorname{Re}f+\gamma\\
\operatorname{Re}f-\gamma & -\operatorname{Im}f
\end{pmatrix}.
$$

Unless a model explicitly overrides them,

$$
z
=
\cos(2k_z)+c
+i\left(
\cos k_x+\cos k_y+\cos k_z-m
\right),
\qquad
w
=
\sin k_x+i\sin k_y.
$$

For the three-link model,

$$
f_{\mathrm{3L}}(z,w)
=
z\left(z^2-w^2\right),
\qquad
c=0.5,\quad m=2.
$$

In [ ]:
three_link_planarity_records = []
for gamma in (0.116, 0.41, 0.5):
    ske_three = NodalSkeleton(
        threelink_bloch_vector(gamma, k_symbols=(kx, ky, kz)),
        k_symbols=(kx, ky, kz),
        dimension=200,
        axis_scale=(1.0, 1.0, 1.5),
    )
    surface_three = ske_three.exceptional_surface_pv
    graph_three = ske_three.skeleton_graph(simplify=True, smooth_epsilon=2)
    planar_three = nx.check_planarity(nx.Graph(graph_three))[0]
    three_link_planarity_records.append(
        (gamma, surface_three, graph_three, planar_three)
    )
    print(f"gamma = {gamma}")
    print_surface_summary("  full surface", surface_three)
    print(
        "  graph_nodes_edges =",
        (graph_three.number_of_nodes(), graph_three.number_of_edges()),
    )
    print("  planar =", planar_three)


In [ ]:
fig = make_subplots(
    rows=2,
    cols=3,
    specs=[
        [{"type": "scene"} for _ in range(3)],
        [{"type": "scene"} for _ in range(3)],
    ],
    horizontal_spacing=0.01,
    vertical_spacing=0.02,
)
for col, (
    gamma,
    surface_three,
    graph_three,
    planar_three,
) in enumerate(three_link_planarity_records, start=1):
    add_surface_trace(fig, surface_three, row=1, col=col, opacity=0.46)
    add_graph_traces(fig, graph_three, row=2, col=col)
style_plotly_scenes(fig, 6, width=1080, height=720).show()


## 8. Generate a small example Yamada dataset

Dataset generation can become expensive very quickly because exact Yamada calculations
grow with graph complexity.

For the user guide, the live calculation is therefore intentionally **small**: only six
simple catalog cases are evaluated. Its purpose is to show the dataset-generation
workflow without making a new user wait for a large exact-computation sweep.

Each generated row stores:

- `graph_name`;
- `varying_params`;
- `yamada`;
- `yamada_sigma`.

A larger precomputed dataset is supplied as
`doc/assets/data/structured_graph_yamada_dataset.csv` at the repository root and is
loaded in the next section.

In [13]:
import csv


# Six deliberately small cases for a fast tutorial run.
DEMO_YAMADA_SWEEPS = {
    "periodic_theta": [
        (2,),
        (3,),
    ],
    "fan": [
        (2,),
        (3,),
    ],
    "ladder": [
        (2,),
        (3,),
    ],
}

planned_demo_row_count = sum(
    len(args_list)
    for args_list in DEMO_YAMADA_SWEEPS.values()
)


def compute_demo_yamada_dataset(output_path=None):
    """Compute a small exact dataset suitable for a quick tutorial run."""
    fieldnames = [
        "graph_name",
        "varying_params",
        "yamada",
        "yamada_sigma",
    ]

    rows = []

    print(
        "Planned tutorial rows =",
        planned_demo_row_count,
    )

    for family_name, args_list in DEMO_YAMADA_SWEEPS.items():
        builder = (
            GRAPH_FAMILY_CATALOG[
                family_name
            ].builder
        )

        for args in args_list:
            graph = builder(*args)

            yamada_expr = sp.expand(
                compute_graph_yamada_polynomial(
                    graph,
                    Y,
                )
            )

            yamada_sigma_expr = (
                laurent_y_to_sigma_polynomial(
                    yamada_expr,
                    Y,
                    sigma,
                )
                .as_expr()
            )

            row = {
                "graph_name": family_name,
                "varying_params": repr(
                    tuple(args)
                ),
                "yamada": sp.sstr(
                    yamada_expr
                ),
                "yamada_sigma": sp.sstr(
                    yamada_sigma_expr
                ),
            }

            rows.append(row)

            print(
                f"{family_name:18s} "
                f"{row['varying_params']:8s} "
                f"{row['yamada']}"
            )

    if output_path is not None:
        output_path = Path(
            output_path
        )

        with output_path.open(
            "w",
            newline="",
            encoding="utf-8",
        ) as handle:
            writer = csv.DictWriter(
                handle,
                fieldnames=fieldnames,
            )
            writer.writeheader()
            writer.writerows(rows)

        print(
            f"\nSaved demo dataset to "
            f"{output_path}"
        )

    return rows


### 8.1 Run the small demonstration

This cell computes only six small graph cases, so it is suitable for a first tutorial
run.

The demo is intentionally separate from the supplied full CSV. Running this cell will
not overwrite `doc/assets/data/structured_graph_yamada_dataset.csv`.

In [ ]:
demo_yamada_dataset = (
    compute_demo_yamada_dataset()
)

print(
    f"\nComputed "
    f"{len(demo_yamada_dataset)} "
    f"tutorial rows."
)

demo_yamada_dataset


## 9. Inspect the supplied structured-graph Yamada dataset

For broader coverage, use the supplied
`structured_graph_yamada_dataset.csv` rather than recomputing a large exact dataset
during the tutorial.

The CSV is distributed under **`doc/assets/data/`** at the repository root. It
contains the precomputed columns

- `graph_name`;
- `varying_params`;
- `yamada`; and
- `yamada_sigma`.

The cell below locates that file, reports its size, and displays the **complete CSV in a
scrollable table**.

In [ ]:
import csv
import html
from IPython.display import HTML


dataset_path = (
    PROJECT_ROOT
    / "doc"
    / "assets"
    / "data"
    / "structured_graph_yamada_dataset.csv"
).resolve()

if not dataset_path.exists():
    raise FileNotFoundError(
        "Could not find the supplied structured Yamada dataset at "
        f"{dataset_path}."
    )

with dataset_path.open(
    newline="",
    encoding="utf-8",
) as handle:
    reader = csv.DictReader(
        handle
    )
    stored_rows = list(
        reader
    )
    stored_columns = list(
        reader.fieldnames or []
    )

print(
    "dataset path =",
    dataset_path,
)
print(
    "stored rows =",
    len(stored_rows),
)
print(
    "columns =",
    stored_columns,
)


def _csv_html_table(
    rows,
    columns,
):
    header = "".join(
        f"<th>{html.escape(column)}</th>"
        for column in columns
    )

    body_rows = []

    for row in rows:
        cells = "".join(
            "<td style='vertical-align:top;"
            "padding:4px 8px;"
            "white-space:pre-wrap;'>"
            f"{html.escape(str(row.get(column, '')))}"
            "</td>"
            for column in columns
        )

        body_rows.append(
            f"<tr>{cells}</tr>"
        )

    return f"""
    <div style="
        max-height:620px;
        overflow:auto;
        border:1px solid #ddd;
    ">
      <table style="
          border-collapse:collapse;
          width:100%;
          font-family:monospace;
          font-size:12px;
      ">
        <thead style="
            position:sticky;
            top:0;
            background:white;
            z-index:1;
        ">
          <tr>{header}</tr>
        </thead>
        <tbody>
          {''.join(body_rows)}
        </tbody>
      </table>
    </div>
    """


display(
    HTML(
        _csv_html_table(
            stored_rows,
            stored_columns,
        )
    )
)


## 10. Use PD codes with your own custom invariants

`KnottedGraph` does not need to implement every invariant internally. A useful extension
pattern is

$$
G\subset\mathbb{R}^3
\longrightarrow
\texttt{PDCode}
\longrightarrow
\text{user-defined invariant}.
$$

This section demonstrates that pattern with **six classical knot embeddings** and two
well-known knot invariants:

1. the **Alexander polynomial** $\Delta_K(t)$; and
2. the **Jones polynomial** $V_K(q)$.

The invariant functions below are **not library functions**. They are deliberately
defined inside this notebook to show how a user can consume the diagram representation
produced by `KnottedGraph`.

### Scope of this example

Alexander and Jones polynomials are invariants of **knots and links**, not arbitrary
spatial graphs with junctions of valence $>2$. Therefore the six examples below are
closed curves represented as one-loop `MultiGraph` embeddings.

For orientation-sensitive calculations we use the `PDCode` **processor object** rather
than only the printed string. After `processor.compute(...)`, it contains

- `processor.crossings`: cyclic crossing data, with positions $0/2$ the over-strand;
- `processor.arcs`: directed arc endpoints; and
- `processor.vertices`: the projected graph vertices.

The printed `V[...] ; X[...]` string is still shown for inspection, but the processor is
the richer interface for building new invariants.

Conventions used below:

- $\Delta_K(t)$ is normalized up to its usual unit ambiguity $\pm t^m$ by shifting the
  lowest exponent to zero and choosing a positive constant term;
- $V_{\bigcirc}(q)=1$, with the Kauffman-bracket substitution $q=A^{-4}$.

Original references:

- J. W. Alexander, *Topological invariants of knots and links*,
  Trans. Amer. Math. Soc. **30** (1928), 275–306,
  https://doi.org/10.1090/S0002-9947-1928-1501429-1
- V. F. R. Jones, *A polynomial invariant for knots via von Neumann algebras*,
  Bull. Amer. Math. Soc. **12** (1985), 103–111,
  https://doi.org/10.1090/S0273-0979-1985-15304-2

In [ ]:
def closed_curve_graph(points):
    """Store one closed curve as a one-loop embedded MultiGraph."""
    points = np.asarray(points, dtype=float)

    if not np.allclose(
        points[0],
        points[-1],
    ):
        points = np.vstack(
            [points, points[0]]
        )
    else:
        points[-1] = points[0]

    graph = nx.MultiGraph()
    graph.add_node(
        0,
        pos=points[0].copy(),
    )
    graph.add_edge(
        0,
        0,
        pts=points,
    )
    return graph


def torus_knot_curve(
    p,
    q,
    *,
    samples=500,
    major_radius=3.0,
    minor_radius=1.0,
):
    """A reproducible positive-(p,q) torus-knot embedding."""
    u = np.linspace(
        0.0,
        2.0 * np.pi,
        samples,
    )

    return np.column_stack(
        [
            (
                major_radius
                + minor_radius
                * np.cos(q * u)
            )
            * np.cos(p * u),
            (
                major_radius
                + minor_radius
                * np.cos(q * u)
            )
            * np.sin(p * u),
            -minor_radius
            * np.sin(q * u),
        ]
    )


def figure_eight_curve(
    *,
    samples=500,
):
    """A standard embedding of the figure-eight knot."""
    u = np.linspace(
        0.0,
        2.0 * np.pi,
        samples,
    )

    return np.column_stack(
        [
            (
                2.0
                + np.cos(2.0 * u)
            )
            * np.cos(3.0 * u),
            (
                2.0
                + np.cos(2.0 * u)
            )
            * np.sin(3.0 * u),
            np.sin(4.0 * u),
        ]
    )


classical_knot_examples = {
    "Unknot": closed_curve_graph(
        torus_knot_curve(1, 1)
    ),
    "Trefoil $3_1$": closed_curve_graph(
        torus_knot_curve(2, 3)
    ),
    "Figure-eight $4_1$": closed_curve_graph(
        figure_eight_curve()
    ),
    "Cinquefoil $5_1$": closed_curve_graph(
        torus_knot_curve(2, 5)
    ),
    "Septafoil $7_1$": closed_curve_graph(
        torus_knot_curve(2, 7)
    ),
    "$8_{19}=T(3,4)$": closed_curve_graph(
        torus_knot_curve(3, 4)
    ),
}

print(
    "created",
    len(classical_knot_examples),
    "embedded knot examples",
)


In [ ]:
fig = make_subplots(
    rows=2,
    cols=3,
    specs=[
        [
            {"type": "scene"},
            {"type": "scene"},
            {"type": "scene"},
        ],
        [
            {"type": "scene"},
            {"type": "scene"},
            {"type": "scene"},
        ],
    ],
    subplot_titles=list(
        classical_knot_examples
    ),
)

for index, graph in enumerate(
    classical_knot_examples.values()
):
    row = index // 3 + 1
    col = index % 3 + 1

    add_graph_traces(
        fig,
        graph,
        row=row,
        col=col,
    )

style_plotly_scenes(
    fig,
    6,
    width=1080,
    height=720,
)

fig.show()


### 10.1 Generate six PD codes with `KnottedGraph`

The next cell performs the projection itself. No PD code is typed by hand.

Each 3D embedding is passed to

```python
processor = PDCode(graph)
pd_code = processor.compute(...)
```

using the same fixed projection angles for reproducibility. The resulting PD strings are
then retained together with their processor objects for the custom calculations below.

In [ ]:
from IPython.display import Markdown


classical_pd_examples = {}

for name, graph in classical_knot_examples.items():
    processor = PDCode(graph)

    pd_code = processor.compute(
        rotation_angles=(
            0.0,
            0.0,
            0.0,
        ),
        rotation_order="ZYX",
    )

    classical_pd_examples[name] = {
        "graph": graph,
        "processor": processor,
        "pd_code": pd_code,
        "crossings": len(
            processor.crossings
        ),
    }


pd_lines = [
    "| Example | Crossings | PD code generated by `KnottedGraph` |",
    "| --- | ---: | --- |",
]

for name, record in classical_pd_examples.items():
    pd_lines.append(
        f"| {name} "
        f"| {record['crossings']} "
        f"| `{record['pd_code']}` |"
    )

display(
    Markdown(
        "\n".join(pd_lines)
    )
)


### 10.2 Define a custom Jones polynomial from the PD representation

This implementation is intentionally self-contained.

For every crossing, `KnottedGraph` has already determined the cyclic order and
over/under information. We perform the two Kauffman smoothings, count the resulting
loops, form the bracket polynomial, and finally normalize by the writhe.

For a diagram with $c$ crossings this direct teaching implementation enumerates $2^c$
states, so it is appropriate for modest examples rather than very large diagrams.

In [19]:
import itertools


A_bracket = sp.Symbol("A")
q_jones = sp.Symbol("q")


class _UnionFind:
    def __init__(
        self,
        items,
    ):
        self.parent = {
            item: item
            for item in items
        }

    def find(
        self,
        item,
    ):
        parent = self.parent[item]

        if parent != item:
            self.parent[item] = (
                self.find(parent)
            )

        return self.parent[item]

    def union(
        self,
        left,
        right,
    ):
        left_root = self.find(left)
        right_root = self.find(right)

        if left_root != right_root:
            self.parent[right_root] = (
                left_root
            )


def _pd_port_for_arc_at(
    processor,
    arc_id,
    endpoint_type,
    endpoint_id,
):
    arc = processor.arcs[arc_id]

    if (
        arc.start_type == endpoint_type
        and arc.start_id == endpoint_id
    ):
        return (
            arc_id,
            "s",
        )

    if (
        arc.end_type == endpoint_type
        and arc.end_id == endpoint_id
    ):
        return (
            arc_id,
            "e",
        )

    raise ValueError(
        "Arc is not incident to the requested "
        "PD endpoint."
    )


def _crossing_ports(
    processor,
    crossing_id,
):
    crossing = (
        processor.crossings[
            crossing_id
        ]
    )

    return [
        _pd_port_for_arc_at(
            processor,
            arc_id,
            "x",
            crossing_id,
        )
        for arc_id
        in crossing.ccw_ordered_arcs
    ]


def _vertex_ports(
    processor,
    vertex_id,
):
    ports = []

    for arc_id, arc in (
        processor.arcs.items()
    ):
        if (
            arc.start_type == "v"
            and arc.start_id == vertex_id
        ):
            ports.append(
                (
                    arc_id,
                    "s",
                )
            )

        if (
            arc.end_type == "v"
            and arc.end_id == vertex_id
        ):
            ports.append(
                (
                    arc_id,
                    "e",
                )
            )

    return ports


def crossing_sign_from_pd(
    processor,
    crossing_id,
):
    """Return +1 or -1 using only oriented PD combinatorics."""
    ordered = (
        processor.crossings[
            crossing_id
        ].ccw_ordered_arcs
    )

    incoming_positions = []

    for position, arc_id in enumerate(
        ordered
    ):
        arc = processor.arcs[
            arc_id
        ]

        if (
            arc.end_type == "x"
            and arc.end_id
            == crossing_id
        ):
            incoming_positions.append(
                position
            )

    over_in = next(
        position
        for position
        in incoming_positions
        if position in (0, 2)
    )
    under_in = next(
        position
        for position
        in incoming_positions
        if position in (1, 3)
    )

    delta = (
        under_in
        - over_in
    ) % 4

    if delta == 1:
        return 1

    if delta == 3:
        return -1

    raise ValueError(
        "Degenerate oriented crossing."
    )


def writhe_from_pd(
    processor,
):
    return sum(
        crossing_sign_from_pd(
            processor,
            crossing_id,
        )
        for crossing_id
        in processor.crossings
    )


def kauffman_bracket_from_pd(
    processor,
    A=A_bracket,
):
    """Compute <D> with <unknot>=1 from a KnottedGraph PD processor."""
    crossing_ids = list(
        processor.crossings
    )

    all_ports = [
        (
            arc_id,
            side,
        )
        for arc_id
        in processor.arcs
        for side
        in ("s", "e")
    ]

    if not all_ports:
        return sp.Integer(1)

    loop_factor = (
        -A**2
        - A**-2
    )
    total = sp.Integer(0)

    for state in itertools.product(
        (0, 1),
        repeat=len(
            crossing_ids
        ),
    ):
        union_find = _UnionFind(
            all_ports
        )

        for arc_id in processor.arcs:
            union_find.union(
                (
                    arc_id,
                    "s",
                ),
                (
                    arc_id,
                    "e",
                ),
            )

        for vertex_id in processor.vertices:
            ports = _vertex_ports(
                processor,
                vertex_id,
            )

            if len(ports) != 2:
                raise ValueError(
                    "This Jones example expects a "
                    "knot/link diagram with valence-2 "
                    f"vertices; vertex {vertex_id} "
                    f"has {len(ports)} half-edges."
                )

            union_find.union(
                ports[0],
                ports[1],
            )

        n_A = 0

        for (
            crossing_id,
            smoothing,
        ) in zip(
            crossing_ids,
            state,
        ):
            ports = _crossing_ports(
                processor,
                crossing_id,
            )

            if smoothing == 0:
                pairs = (
                    (0, 3),
                    (1, 2),
                )
                n_A += 1
            else:
                pairs = (
                    (0, 1),
                    (2, 3),
                )

            for left, right in pairs:
                union_find.union(
                    ports[left],
                    ports[right],
                )

        loop_count = len(
            {
                union_find.find(port)
                for port
                in all_ports
            }
        )

        n_B = (
            len(crossing_ids)
            - n_A
        )

        total += (
            A**(
                n_A
                - n_B
            )
            * loop_factor**(
                loop_count
                - 1
            )
        )

    return sp.expand(total)


def jones_polynomial_from_pd(
    processor,
    q=q_jones,
):
    """Jones polynomial with V_unknot(q)=1 and q=A^(-4)."""
    bracket = (
        kauffman_bracket_from_pd(
            processor,
            A=A_bracket,
        )
    )

    writhe = writhe_from_pd(
        processor
    )

    normalized_A = sp.expand(
        (
            -A_bracket**3
        )**(-writhe)
        * bracket
    )

    result = sp.Integer(0)

    for term in sp.Add.make_args(
        normalized_A
    ):
        coefficient, exponent = (
            term.as_coeff_exponent(
                A_bracket
            )
        )
        exponent = int(exponent)

        if exponent % 4 != 0:
            raise ValueError(
                "This compact converter is written "
                "for one-component knot examples; "
                f"encountered A^{exponent}."
            )

        result += (
            coefficient
            * q**(
                -exponent // 4
            )
        )

    return sp.expand(result)


### 10.3 Define a custom Alexander polynomial from the same PD representation

For the Alexander polynomial we construct a Wirtinger/Alexander matrix directly from
the oriented diagram.

At a crossing, the over-strand remains in one Wirtinger generator while the under-strand
is split into incoming and outgoing generators. Deleting one row and one column gives
the usual Alexander minor.

The result is defined only up to multiplication by $\pm t^m$; the helper below fixes a
consistent representative for display.

In [20]:
t_alexander = sp.Symbol("t")


def _wirtinger_generator_map(
    processor,
):
    arc_ids = list(
        processor.arcs
    )

    if not arc_ids:
        return {}, 0

    union_find = _UnionFind(
        arc_ids
    )

    for vertex_id in processor.vertices:
        incident = []

        for arc_id, arc in (
            processor.arcs.items()
        ):
            if (
                (
                    arc.start_type
                    == "v"
                    and arc.start_id
                    == vertex_id
                )
                or (
                    arc.end_type
                    == "v"
                    and arc.end_id
                    == vertex_id
                )
            ):
                incident.append(
                    arc_id
                )

        incident = list(
            dict.fromkeys(
                incident
            )
        )

        if len(incident) == 2:
            union_find.union(
                incident[0],
                incident[1],
            )
        elif len(incident) == 1:
            pass
        else:
            raise ValueError(
                "This Alexander example expects "
                "a knot/link diagram with "
                "valence-2 vertices."
            )

    for crossing in (
        processor.crossings.values()
    ):
        ordered = (
            crossing.ccw_ordered_arcs
        )

        union_find.union(
            ordered[0],
            ordered[2],
        )

    root_to_generator = {}
    arc_to_generator = {}

    for arc_id in arc_ids:
        root = union_find.find(
            arc_id
        )

        root_to_generator.setdefault(
            root,
            len(
                root_to_generator
            ),
        )

        arc_to_generator[
            arc_id
        ] = root_to_generator[
            root
        ]

    return (
        arc_to_generator,
        len(root_to_generator),
    )


def _crossing_wirtinger_data(
    processor,
    crossing_id,
    arc_to_generator,
):
    ordered = (
        processor.crossings[
            crossing_id
        ].ccw_ordered_arcs
    )

    direction = []

    for arc_id in ordered:
        arc = processor.arcs[
            arc_id
        ]

        if (
            arc.start_type == "x"
            and arc.start_id
            == crossing_id
        ):
            direction.append(
                "out"
            )
        elif (
            arc.end_type == "x"
            and arc.end_id
            == crossing_id
        ):
            direction.append(
                "in"
            )
        else:
            raise ValueError(
                "Inconsistent crossing/arc data."
            )

    over_generator = (
        arc_to_generator[
            ordered[0]
        ]
    )

    under_in = next(
        arc_to_generator[
            ordered[position]
        ]
        for position
        in (1, 3)
        if direction[
            position
        ] == "in"
    )

    under_out = next(
        arc_to_generator[
            ordered[position]
        ]
        for position
        in (1, 3)
        if direction[
            position
        ] == "out"
    )

    return (
        crossing_sign_from_pd(
            processor,
            crossing_id,
        ),
        over_generator,
        under_in,
        under_out,
    )


def _normalize_alexander(
    expression,
    t=t_alexander,
):
    expression = sp.expand(
        expression
    )

    if expression == 0:
        return expression

    exponents = [
        int(
            term.as_powers_dict().get(
                t,
                0,
            )
        )
        for term
        in sp.Add.make_args(
            expression
        )
    ]

    expression = sp.expand(
        expression
        * t**(
            -min(exponents)
        )
    )

    constant_term = (
        expression.coeff(
            t,
            0,
        )
    )

    if (
        constant_term
        .could_extract_minus_sign()
    ):
        expression = (
            -expression
        )

    return sp.expand(
        expression
    )


def alexander_polynomial_from_pd(
    processor,
    t=t_alexander,
):
    """Alexander polynomial from a one-component KnottedGraph PD diagram."""
    crossing_count = len(
        processor.crossings
    )

    if crossing_count == 0:
        return sp.Integer(1)

    (
        arc_to_generator,
        generator_count,
    ) = _wirtinger_generator_map(
        processor
    )

    if (
        generator_count
        != crossing_count
    ):
        raise ValueError(
            "This compact demonstration expects "
            "a regular one-component knot diagram. "
            f"Found {crossing_count} crossings "
            f"and {generator_count} "
            "Wirtinger generators."
        )

    matrix = sp.zeros(
        crossing_count,
        generator_count,
    )

    for row, crossing_id in enumerate(
        processor.crossings
    ):
        (
            sign,
            over,
            under_in,
            under_out,
        ) = _crossing_wirtinger_data(
            processor,
            crossing_id,
            arc_to_generator,
        )

        matrix[
            row,
            over,
        ] += (
            1
            - t
        )

        if sign > 0:
            matrix[
                row,
                under_out,
            ] += t
            matrix[
                row,
                under_in,
            ] += -1
        else:
            matrix[
                row,
                under_in,
            ] += t
            matrix[
                row,
                under_out,
            ] += -1

    minor = matrix[
        :-1,
        :-1,
    ]

    return _normalize_alexander(
        sp.det(minor),
        t=t,
    )


### 10.4 Evaluate both custom invariants on all six PD codes

The values below are calculated **from the PD representation generated above**.

The assertions make these examples executable checks: if the projection convention or
custom invariant implementation is accidentally changed, the cell fails instead of
silently displaying a different polynomial.

In [ ]:
expected_custom_invariants = {
    "Unknot": {
        "crossings": 0,
        "alexander": sp.Integer(1),
        "jones": sp.Integer(1),
    },
    "Trefoil $3_1$": {
        "crossings": 3,
        "alexander": (
            t_alexander**2
            - t_alexander
            + 1
        ),
        "jones": (
            q_jones
            + q_jones**3
            - q_jones**4
        ),
    },
    "Figure-eight $4_1$": {
        "crossings": 4,
        "alexander": (
            t_alexander**2
            - 3 * t_alexander
            + 1
        ),
        "jones": (
            q_jones**2
            - q_jones
            + 1
            - q_jones**-1
            + q_jones**-2
        ),
    },
    "Cinquefoil $5_1$": {
        "crossings": 5,
        "alexander": (
            t_alexander**4
            - t_alexander**3
            + t_alexander**2
            - t_alexander
            + 1
        ),
        "jones": (
            q_jones**2
            + q_jones**4
            - q_jones**5
            + q_jones**6
            - q_jones**7
        ),
    },
    "Septafoil $7_1$": {
        "crossings": 7,
        "alexander": (
            t_alexander**6
            - t_alexander**5
            + t_alexander**4
            - t_alexander**3
            + t_alexander**2
            - t_alexander
            + 1
        ),
        "jones": (
            q_jones**3
            + q_jones**5
            - q_jones**6
            + q_jones**7
            - q_jones**8
            + q_jones**9
            - q_jones**10
        ),
    },
    "$8_{19}=T(3,4)$": {
        "crossings": 8,
        "alexander": (
            t_alexander**6
            - t_alexander**5
            + t_alexander**3
            - t_alexander
            + 1
        ),
        "jones": (
            q_jones**3
            + q_jones**5
            - q_jones**8
        ),
    },
}


custom_invariant_rows = []

for name, record in (
    classical_pd_examples.items()
):
    processor = record[
        "processor"
    ]

    alexander = (
        alexander_polynomial_from_pd(
            processor,
            t=t_alexander,
        )
    )

    jones = (
        jones_polynomial_from_pd(
            processor,
            q=q_jones,
        )
    )

    expected = (
        expected_custom_invariants[
            name
        ]
    )

    assert (
        record["crossings"]
        == expected["crossings"]
    )

    assert sp.simplify(
        alexander
        - expected["alexander"]
    ) == 0

    assert sp.simplify(
        jones
        - expected["jones"]
    ) == 0

    custom_invariant_rows.append(
        {
            "name": name,
            "crossings": record[
                "crossings"
            ],
            "pd_code": record[
                "pd_code"
            ],
            "alexander": alexander,
            "jones": jones,
        }
    )


result_lines = [
    "| Example | $c$ | Alexander $\\Delta_K(t)$ | Jones $V_K(q)$ |",
    "| --- | ---: | --- | --- |",
]

for row in custom_invariant_rows:
    result_lines.append(
        f"| {row['name']} "
        f"| {row['crossings']} "
        f"| ${sp.latex(row['alexander'])}$ "
        f"| ${sp.latex(row['jones'])}$ |"
    )

display(
    Markdown(
        "\n".join(
            result_lines
        )
    )
)

print(
    "all six custom-invariant checks passed"
)


### 10.5 The invariant should not depend on which regular projection is used

A useful check for a custom PD-code consumer is to rotate the **same spatial embedding**,
obtain different projections, and verify that the invariant remains unchanged.

Below the trefoil is evaluated from four projections. The PD strings may be ordered
differently, but the Alexander and Jones polynomials agree.

In [ ]:
trefoil_graph = (
    classical_knot_examples[
        "Trefoil $3_1$"
    ]
)

trefoil_angles = [
    (0.0, 0.0, 0.0),
    (19.0, 27.0, 0.0),
    (43.0, 11.0, 0.0),
    (71.0, 38.0, 0.0),
]

trefoil_projection_results = []

for angles in trefoil_angles:
    processor = PDCode(
        trefoil_graph
    )

    pd_code = processor.compute(
        rotation_angles=angles,
        rotation_order="ZYX",
    )

    alexander = (
        alexander_polynomial_from_pd(
            processor
        )
    )

    jones = (
        jones_polynomial_from_pd(
            processor
        )
    )

    trefoil_projection_results.append(
        (
            angles,
            len(
                processor.crossings
            ),
            pd_code,
            alexander,
            jones,
        )
    )


reference_alexander = (
    trefoil_projection_results[
        0
    ][3]
)
reference_jones = (
    trefoil_projection_results[
        0
    ][4]
)

for (
    _,
    _,
    _,
    alexander,
    jones,
) in trefoil_projection_results:
    assert sp.simplify(
        alexander
        - reference_alexander
    ) == 0

    assert sp.simplify(
        jones
        - reference_jones
    ) == 0


projection_lines = [
    "| Rotation angles | Crossings | Alexander | Jones |",
    "| --- | ---: | --- | --- |",
]

for (
    angles,
    crossings,
    _,
    alexander,
    jones,
) in trefoil_projection_results:
    projection_lines.append(
        f"| `{angles}` "
        f"| {crossings} "
        f"| ${sp.latex(alexander)}$ "
        f"| ${sp.latex(jones)}$ |"
    )

display(
    Markdown(
        "\n".join(
            projection_lines
        )
    )
)

print(
    "trefoil projection-invariance check passed"
)


### 10.6 Template for another user-defined PD-code invariant

The same pattern can be reused for other diagram-based quantities:

```python
def my_invariant_from_pd(processor):
    for crossing_id, crossing in processor.crossings.items():
        ordered_arcs = crossing.ccw_ordered_arcs
        # ordered_arcs[0] and ordered_arcs[2] form the over-strand

    for arc_id, arc in processor.arcs.items():
        # arc.start_type / arc.start_id
        # arc.end_type   / arc.end_id
        # describe how diagram pieces are connected
        ...

    return invariant
```

This keeps the responsibilities separated:

$$
\texttt{KnottedGraph}
:
G\subset\mathbb R^3
\rightarrow
\operatorname{PD}(G),
$$

while user code handles

$$
\operatorname{PD}(G)
\rightarrow
I(G).
$$

For an invariant defined specifically for general spatial graphs rather than knots/links,
the same PD representation can be used, but the local rules at vertices of valence
$>2$ must be those of that invariant.

## 11. Use exact computation to discover a mathematical pattern

A productive workflow for a new graph family is:

1. define the family and its discrete parameters;
2. compute exact invariants over the largest practical range;
3. inspect degrees, factors, coefficients, and possible recurrences;
4. formulate a conjecture from the exact data; and
5. prove or independently verify the recurrence or closed form.

Finite computation gives evidence and candidate structure. It does not replace a proof.

## 12. Periodic theta graphs: computing and verifying a Yamada closed form

This example shows how `KnottedGraph` can be used on a parameterized graph family to move from **exact computation** to a **symbolic formula** and then verify that formula independently.

`build_periodic_theta_graph(s)` constructs the periodic theta graph $\theta_{s,P}$ from two pole vertices and $s$ midpoint vertices $m_0,\ldots,m_{s-1}$. Each midpoint is connected to both poles, and the midpoint vertices are connected cyclically:

$$
V(\theta_{s,P})=\{u,v,m_0,\ldots,m_{s-1}\},
$$

$$
E(\theta_{s,P})=
\{u m_i,\;v m_i,\;m_i m_{i+1}\}_{i\;\mathrm{mod}\;s}.
$$

Hence

$$
|V|=s+2,
\qquad
|E|=3s.
$$

Introduce the standard Yamada combination

$$
\sigma=Y+1+Y^{-1}.
$$

The predicted Yamada polynomial for the family is

$$
\boxed{
\Upsilon(\theta_{s,P};Y)
=
\frac{
(\sigma^2-\sigma+1)^s
+\sigma(2-\sigma)^s
+\sigma(-\sigma)^s
+\sigma^2-\sigma-1
}{\sigma+1}
},
\qquad s\ge2.
$$

Although this expression is written as a quotient, the numerator is divisible by $\sigma+1$, so the result is an ordinary polynomial in $\sigma$. The cells below verify the formula directly with `KnottedGraph`, check an independent planar-dual route with NetworkX/SymPy, and then generate a table of exact Yamada polynomials for $2\le s\le18$.


In [ ]:
from knotted_graph.applications.mathematical import build_periodic_theta_graph
from knotted_graph.invariants.yamada import (
    compute_graph_yamada_polynomial,
    laurent_y_to_sigma_polynomial,
)

sigma_periodic = sp.Symbol("sigma")
q_periodic = sp.Symbol("q")


def predicted_periodic_theta_yamada_sigma(s, sigma=sigma_periodic):
    """Predicted Yamada polynomial Upsilon(theta_{s,P}; Y), written in sigma."""
    if s < 2:
        raise ValueError("This family formula is used for s >= 2.")

    # Explicit prediction:
    # Upsilon(theta_{s,P};Y) =
    # [
    #   (sigma^2-sigma+1)^s
    #   + sigma(2-sigma)^s
    #   + sigma(-sigma)^s
    #   + sigma^2-sigma-1
    # ] / (sigma+1),
    # with sigma = Y + 1 + Y^(-1).
    numerator = sp.expand(
        (sigma**2 - sigma + 1) ** s
        + sigma * (2 - sigma) ** s
        + sigma * (-sigma) ** s
        + sigma**2
        - sigma
        - 1
    )

    quotient, remainder = sp.div(
        sp.Poly(numerator, sigma),
        sp.Poly(sigma + 1, sigma),
    )
    assert remainder.is_zero
    return quotient


for s in range(2, 19):
    predicted = predicted_periodic_theta_yamada_sigma(s)
    assert predicted.degree() == 2 * s - 1
    assert predicted.LC() == 1

print("Predicted formula is polynomial, monic, and degree 2s-1 for s=2,...,18")


### 12.1 Verify the prediction

We use two checks that start from different descriptions of the same family.

**Direct library check.** For several values of $s$, `KnottedGraph` computes $\Upsilon(\theta_{s,P};Y)$ directly from the graph. We then rewrite the result in $\sigma=Y+1+Y^{-1}$ and ask SymPy to verify

$$
\Upsilon_{\mathrm{KnottedGraph}}(\theta_{s,P};Y)
-
\frac{
(\sigma^2-\sigma+1)^s
+\sigma(2-\sigma)^s
+\sigma(-\sigma)^s
+\sigma^2-\sigma-1
}{\sigma+1}
=0.
$$

**Independent planar-dual check.** For $s\ge3$, the planar dual of $\theta_{s,P}$ is the ordinary $s$-prism graph. Its chromatic polynomial can be written explicitly as

$$
\chi_s(q)
=
(q^2-3q+3)^s
+(q-1)(3-q)^s
+(q-1)(1-q)^s
+q^2-3q+1.
$$

NetworkX is used to check this chromatic-polynomial expression for small $s$. Substituting $q=\sigma+1$ into the planar duality relation then gives exactly the same explicit Yamada prediction above. This provides a verification route that is separate from the direct Yamada recursion.


In [ ]:
# 1) Direct KnottedGraph computation versus the explicit prediction.
direct_periodic_checks = {}
for s in range(2, 7):
    graph = build_periodic_theta_graph(s)

    yamada_y = sp.expand(
        compute_graph_yamada_polynomial(graph, Y)
    )
    yamada_sigma = sp.Poly(
        laurent_y_to_sigma_polynomial(
            yamada_y,
            Y,
            sigma_periodic,
        ).as_expr(),
        sigma_periodic,
    )

    predicted_sigma = predicted_periodic_theta_yamada_sigma(s)
    difference = sp.expand(
        yamada_sigma.as_expr() - predicted_sigma.as_expr()
    )
    assert difference == 0, (s, difference)

    direct_periodic_checks[s] = yamada_sigma
    print(f"s={s:2d}: KnottedGraph == explicit Yamada prediction  PASS")


# 2) Independent chromatic-polynomial check for the planar dual prism.
def prism_chromatic_polynomial_formula(s, q=q_periodic):
    return sp.expand(
        (q**2 - 3*q + 3) ** s
        + (q - 1) * (3 - q) ** s
        + (q - 1) * (1 - q) ** s
        + q**2
        - 3 * q
        + 1
    )


for s in (3, 4, 5):
    prism = nx.circular_ladder_graph(s)
    chromatic = nx.chromatic_polynomial(prism)
    chromatic_symbol = next(iter(chromatic.free_symbols))
    chromatic_q = sp.expand(
        chromatic.subs(chromatic_symbol, q_periodic)
    )
    expected_q = prism_chromatic_polynomial_formula(s)
    assert sp.expand(chromatic_q - expected_q) == 0
    print(f"s={s:2d}: independent prism chromatic formula          PASS")


# 3) SymPy verifies that the planar-dual expression reduces to the same
#    explicit predicted Yamada polynomial for the full displayed range.
for s in range(3, 19):
    dual_yamada_sigma = sp.cancel(
        prism_chromatic_polynomial_formula(s, q_periodic).subs(
            q_periodic,
            sigma_periodic + 1,
        )
        / (sigma_periodic + 1)
    )

    explicit_prediction = predicted_periodic_theta_yamada_sigma(s).as_expr()
    difference = sp.expand(dual_yamada_sigma - explicit_prediction)
    assert difference == 0, (s, difference)

print("SymPy independent-form check passed for s=3,...,18")


### 12.2 Exact Yamada values for the family

Once the formula has been verified, it can be used to generate exact values efficiently for larger $s$. The table below lists

$$
\Upsilon(\theta_{s,P};Y)
=
\frac{
(\sigma^2-\sigma+1)^s
+\sigma(2-\sigma)^s
+\sigma(-\sigma)^s
+\sigma^2-\sigma-1
}{\sigma+1}
$$

as an expanded polynomial in $\sigma$ for $2\le s\le18$.


In [ ]:
from IPython.display import Markdown

periodic_theta_table = []
for s in range(2, 19):
    yamada_sigma = predicted_periodic_theta_yamada_sigma(s)
    periodic_theta_table.append(
        {
            "s": s,
            "degree": yamada_sigma.degree(),
            "polynomial": sp.expand(yamada_sigma.as_expr()),
        }
    )

periodic_table_lines = [
    "| $s$ | degree | $\\Upsilon(\\theta_{s,P};Y)$ as a polynomial in $\\sigma$ |",
    "| ---: | ---: | --- |",
]
for row in periodic_theta_table:
    periodic_table_lines.append(
        f"| {row['s']} | {row['degree']} | $"
        f"{sp.latex(row['polynomial'])}$ |"
    )

display(Markdown("\n".join(periodic_table_lines)))


In [ ]:
print("Optional copy-ready LaTeX table rows:\n")
for row in periodic_theta_table:
    latex_poly = sp.latex(row["polynomial"])
    print(
        f"${row['s']}$ & "
        f"$\\displaystyle {latex_poly}$ \\\\" 
    )


For projection-level diagnostics, state expansions, and geometric robustness tests, continue to
**[Advanced & Reproduction](../03_advanced_and_reproduction.ipynb)**.